# F11 · TwinCheck demo with the image card (Feba)

**Runtime → T4 GPU**, switch on **Notebook access** for `GITHUB_TOKEN` and `HF_TOKEN` in the 🔑 panel, then run the cells in order.

1. Setup: Drive, repo, packages. The demo uses the trained verdict head and the fast text mode.
2. Tests.
3. Build the F3 JPEG settings pool once (saved on Drive as `models/F11/jpeg_pool.json`, ~2 min).
4. Preprocessing check: 10 cached and 10 raw GenImage test images vs `results/F10/preds_vit_b16_s42_check.csv`; pushes `results/F11/preprocess_check.csv` (ends with `PUSHED`).
5. Launch the demo (public `gradio.live` link). This cell keeps running while the demo is open.

Image examples: put the pictures and an `examples.json` in `MyDrive/TwinCheck/demo_images/` (format in `src/demo/vision.py`). Without that folder the image buttons are hidden; uploading works anyway.


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/TwinCheck'
REPO = '/content/twincheck'
BRANCH = 'feba/F11-demo-vision'
CKPT = f'{DRIVE}/models/F5/vit_b16_s42.pt'
POOL = f'{DRIVE}/models/F11/jpeg_pool.json'
CACHE = f'{DRIVE}/data/processed/vision/genimage_debiased96_jpeg.pkl'
DEMO_IMAGES = f'{DRIVE}/demo_images'
if not os.path.exists(REPO):
    !git clone -q https://github.com/himaallu/Group-Project.git {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
!git config user.name "Rathi"
!git config user.email "335395537+rathiabeth-sudo@users.noreply.github.com"
GH = userdata.get('GITHUB_TOKEN').strip()
!git remote set-url --push origin https://{GH}@github.com/himaallu/Group-Project.git
!pip install -q -U gradio rank_bm25 faiss-cpu
!python -m spacy download -q en_core_web_sm
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
os.environ.update(TWINCHECK_NLI='MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7',
                  TWINCHECK_VERDICT='models/verdict_twin_s42', TWINCHECK_EXTRACTOR='split',
                  TWINCHECK_EXPLAINER='template', TWINCHECK_SPACY='en_core_web_sm')
print({p: os.path.exists(p) for p in (CKPT, CACHE, DEMO_IMAGES)})


In [ ]:
!python -m pytest -q tests/demo

In [ ]:
os.makedirs(os.path.dirname(POOL), exist_ok=True)
if not os.path.exists(POOL):
    !python -m src.vision.infer --build-pool --drive {DRIVE} --jpeg-pool {POOL}
print(os.path.exists(POOL))


In [ ]:
!python -m src.vision.infer --ckpt {CKPT} --jpeg-pool {POOL} --verify-cache {CACHE} --verify-raw --drive {DRIVE} --n 10 --out results/F11/preprocess_check.csv
!git add results/F11/preprocess_check.csv && git commit -q -m "[F11] Preprocessing check: 10 cached + 10 raw test images vs F10" && git push -q origin {BRANCH} && echo PUSHED


In [ ]:
!python -m src.demo.app --share --vision-ckpt {CKPT} --jpeg-pool {POOL} --demo-images {DEMO_IMAGES}